# 13. MPI on CIRRUS

← [Functions](12-functions.ipynb) · next: [LLM services](14-llm.ipynb)

Yes, you can run MPI on CIRRUS. Read [page 1](01-orientation.ipynb) again first:
**for a large tightly-coupled job, Derecho is the right machine** — its
interconnect and its scheduler exist for exactly that, and Kubernetes is not
going to beat them.

What the MPI support here is genuinely good for is MPI work that needs to live
*next to something else*: a CI pipeline that runs a multi-node test on every pull
request, a distributed training job triggered by an API call, a workflow that
mixes a short parallel step with services.

The mechanism is the **[Kubeflow MPI Operator](https://github.com/kubeflow/mpi-operator)**,
which adds an `MPIJob` resource. You describe the job; the operator creates the
launcher and worker pods, distributes SSH keys between them, and cleans up.

```yaml
apiVersion: kubeflow.org/v2beta1
kind: MPIJob
metadata:
  name: pi
spec:
  slotsPerWorker: 1
  mpiReplicaSpecs:
    Launcher:                       # runs mpirun; your program lives here
      replicas: 1
      template:
        spec:
          containers:
            - image: docker.io/mpioperator/mpi-pi:openmpi
              name: mpi-launcher
              command: ["mpirun"]
              args: ["-n", "2", "/home/mpiuser/pi"]
    Worker:                         # runs sshd and waits to be told what to do
      replicas: 2
      template:
        spec:
          containers:
            - image: docker.io/mpioperator/mpi-pi:openmpi
              name: mpi-worker
              command: ["/usr/sbin/sshd", "-De", "-f", "/home/mpiuser/.sshd_config"]
```


```bash
kubectl apply -f mpijob.yaml
kubectl get mpijob pi -w
kubectl logs -l training.kubeflow.org/job-name=pi -c mpi-launcher
kubectl delete mpijob pi
```

### The four things that actually go wrong

* **SSH is the whole mechanism, and most failures are SSH failures.** The
  launcher `ssh`es into the workers to start ranks. So workers must run `sshd` as
  their main process, the image must have host keys, and `sshAuthMountPath`
  (default `/root/.ssh`) must be where the image expects them. `Connection timed
  out` means the worker is not running `sshd`; `Permission denied (publickey)`
  means the keys are not where they should be; `no hostkeys available` means the
  image is missing `/etc/ssh/ssh_host_*_key`.
* **The image must contain MPI.** `command not found` on `mpirun` is not a
  Kubernetes problem. Start from `mpioperator/mpi-pi:openmpi` or
  `mpioperator/mpi-horovod-mnist` to prove the plumbing, then build your own on
  OpenMPI, Intel MPI or MVAPICH.
* **Read the launcher's logs, not the workers'.** `mpirun`'s output — and your
  program's — appears on the launcher. The workers mostly log `sshd`.
* **`Pending` means resources, as always.** `kubectl describe mpijob` and
  `kubectl describe pod <worker>`; usually the total request across all workers
  exceeds the quota.

Two flags worth knowing: `slotsPerWorker` should match the ranks you want per pod
(for GPU work, the GPU count), and `runPolicy.cleanPodPolicy: Running` keeps the
pods after completion so you can inspect them — then delete the MPIJob by hand.

For GPUs, request them like any other resource, and use the framework's own
launcher inside `mpirun`:

```yaml
          resources:
            limits:
              nvidia.com/gpu: 1
```


For large shared datasets, mount them rather than copying — see
[page 7](07-storage.ipynb).

Submitting an MPIJob from a GitHub Actions workflow is the CI pattern, and it is
the same submit / wait / collect / clean-up shape as
[page 8](08-github-actions.ipynb#running-against-the-cluster-from-a-workflow).

**Access:** a namespace and Kubernetes API access to it
([page 1](01-orientation.ipynb#kubectl-from-your-own-machine)), since an `MPIJob`
is submitted through the API. Say how many nodes and GPUs you are aiming for in
the ticket, so the quota fits.

Docs: <https://ncar-hpc-docs.readthedocs.io/en/latest/compute-systems/cirrus/guides/13-mpi/mpi/>

## Check yourself

1. Your MPIJob fails with `Permission denied (publickey)`. Which pod's
   configuration is wrong?
2. Where does `mpirun`'s output appear: the launcher or the workers?
3. You have a 500-node coupled model run. Is CIRRUS the right platform? Why not?

---

← [12. Functions](12-functions.ipynb) · next: [14. LLM services](14-llm.ipynb)